# 71 · Governance & quality — a live Soda data-quality scan, via Trino

**This is the quality tier: an *independent contract* over the published marts.**
The transform notebook (`40`) built the seven `iceberg.dbt.mart_*` tables and ran **dbt's own tests**
*as part of the build* — uniqueness, not-null, range — so a failing test fails `dbt build` and the mart
is never published. This notebook runs a **separate** quality engine, **Soda**, *after* publication:
it re-asserts a declarative contract against the live tables and reports **pass / warn / fail per
check**, with the measured value next to each threshold.

That is the thesis of the quality tier:

> **dbt tests gate the build from the inside; Soda scans the published result from the outside. Two
> independent contracts over the same data — if a mart drifts after it ships, the outside contract is
> what still catches it.**

### Where it sits

```
  consumers          Lightdash / Cube (41) / MetricFlow          <- read the marts
  ---------------------------------------------------------------
  quality (Soda)     independent contract scan  ->  pass/fail     <- THIS notebook
  ---------------------------------------------------------------
  transform (dbt)    staging -> marts, dbt tests gate the build   <- nb 40 built these
  ---------------------------------------------------------------
  lakehouse          Iceberg tables on Nessie / MinIO (nb 11)
```

Soda sits **beside** the marts, not inside the pipeline that builds them. In the lab this is exactly
how the Dagster `soda_scan_op` runs it (L5 Slice C) — the same `configuration.yml` and `checks/` we use
below, shelling out to an isolated venv on a schedule and emitting each outcome to DataHub's Assertions
tab. Here we drive the **same scan from the Soda Python API** so you can watch it run and read the
result frame.

> **Read-only, throughout.** A Soda scan issues `SELECT count(*)`, `min(...)`, `max(...)`,
> `missing_count(...)` queries and reads the answers back. It never writes to Iceberg or Nessie. As in
> `40`/`22`, there is no cleanup section — nothing is created. **A check that *fails* is a valid
> result, not an error:** that is the entire point of a contract scan, and this notebook renders a
> failing check as a red row in a table, never as a crashed cell.


## Data-quality gates — Soda vs dbt tests vs Great Expectations

Three tools in this stack assert data quality, and they are complementary, not redundant:

| tool | where it runs | what it is | notebook |
|------|---------------|------------|----------|
| **dbt tests** | *inside* `dbt build` | in-transform gate — a failing test fails the build, the mart never ships | `40` (the marts) |
| **Soda** | *after* publication, on a schedule | an **independent declarative contract** re-checked against the live marts + gold sources | **this one** |
| **Great Expectations** | *after* publication, expectation suites | richer per-column expectations + a data-docs site; heavier to author | (GE tier) |

**Soda's declarative check language (SodaCL)** covers the high-value data-quality dimensions with one
line each:

- **volume** — `row_count > 0` (“is the data actually there?” — catches empty tables and failed loads)
- **missing / completeness** — `missing_count(col) = 0`, `missing_percent(col) < 100`
- **validity / range** — `min(col) >= 0`, `max(col) <= 100` (values inside the domain they claim)
- **uniqueness** — `duplicate_count(col) = 0`
- **freshness** — `freshness(ts_col) < 1d`
- **distribution / anomaly** — reference-dataset and anomaly checks (Soda's richer tiers)

The lab's checks live in `soda/checks/` — `baseline.yml` (a row-count floor fanned across *every*
table), plus `music.yml` / `health.yml` (strict mart contracts), `music_silver.yml` /
`health_gold.yml` (richer bounds on the silver/gold sources). We run the **baseline contract over the
marts** below, and show a richer file as an illustration of the language.


## Setup

`soda-core-trino` (Soda Core plus its Trino connector) is **not** in the singleuser base image, so we
install it here — the same `%pip install` pattern the rest of the wave uses. It pulls in `soda-core`
and the Trino driver. `polars`, used to render the result frame, already ships in the image.

We also pin **`setuptools<81`** alongside it: Soda Core 3.5.x still imports `distutils`, which was
removed from the Python 3.12 standard library — `setuptools<81` vendors a `distutils` shim (activated
by its startup `.pth`) that supplies it. Without the pin, `from soda.scan import Scan` raises
`ModuleNotFoundError: No module named 'distutils'` on a 3.12 kernel.


In [1]:
%pip install -q soda-core-trino "setuptools<81"


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## The Soda data source — configuration, in-cluster and passwordless

Soda needs a **data source** definition (the warehouse it scans). We build it here from the exact shape
of the committed `soda/configuration.yml`, so this notebook drives the *same* connection the Dagster
`soda_scan_op` uses.

The one subtlety worth understanding is **auth**. Soda's Trino connector defaults to
`BasicAuthentication`, and the current `trino` client **refuses to send Basic auth over plaintext
`http`** — it raises `TLS/SSL is required for authentication` *before* a request even leaves the pod.
But the mesh Trino runs **no authenticator** at all. So we set **`auth_type: NoAuthentication`**: Soda
sends **no `Authorization` header**, and we point it at **`trino-noauth`** — a ClusterIP proxy that
forwards only `X-Trino-User: dbt` (the same no-auth path dbt writes the marts through; Ranger
authorizes `dbt` on `iceberg.dbt`). Pointing at `trino.data-mesh` directly would 401; the proxy is the
supported path, and it has **no credentials** of its own — which is exactly why this notebook is
validated **in-cluster**.

Connection is **env-driven** with **in-cluster committed defaults**: `SODA_TRINO_HOST` defaults to the
`trino-noauth` service DNS. We keep the port a literal `8080` in the YAML on purpose — Kubernetes
auto-injects a `TRINO_PORT=tcp://…` service variable that would shadow a port read from the
environment, so we do not read one. Nothing here is secret (the proxy takes no password), so the
committed default is safe to ship.


In [2]:
import os

# Soda data source, built from the committed soda/configuration.yml shape. Host is env-driven with the
# in-cluster trino-noauth default; port is a literal 8080 (NOT read from env -- k8s injects a colliding
# TRINO_PORT=tcp://... service var). No password: trino-noauth strips auth, so Trino sees only the user.
DATA_SOURCE = "weyland"
TRINO_HOST = os.environ.get("SODA_TRINO_HOST", "trino-noauth.data-mesh.svc.cluster.local")

CONFIG_YAML = f"""
data_source {DATA_SOURCE}:
  type: trino
  host: {TRINO_HOST}
  port: 8080
  http_scheme: http
  auth_type: NoAuthentication
  username: dbt
  catalog: iceberg
  schema: dbt
"""

# prove the shape without echoing anything sensitive (there is nothing sensitive -- no password exists)
print(f"data source : {DATA_SOURCE}")
print(f"warehouse   : trino (iceberg.dbt), via the trino-noauth proxy, plaintext + Istio mTLS on the hop")
print(f"host from   : SODA_TRINO_HOST env (committed default = in-cluster trino-noauth service DNS)")


data source : weyland
warehouse   : trino (iceberg.dbt), via the trino-noauth proxy, plaintext + Istio mTLS on the hop
host from   : SODA_TRINO_HOST env (committed default = in-cluster trino-noauth service DNS)


## The contract — what a Soda checks file looks like

This is the lab's `soda/checks/baseline.yml`, verbatim. It is the single highest-value data-quality
assertion — **“is the data actually there?”** — expressed **once** and fanned across **every** table in
the data source with SodaCL's `for each dataset` construct, rather than hand-writing a block per table:

```yaml
# B80 breadth -- the ONE baseline data-quality check that applies to every table, generated dynamically
# so we don't hand-write a block per table. `for each dataset` + `include %` fans `row_count > 0` across
# EVERY table in the scan's data-source schema. "Is the data actually there?" is the single highest-value
# DQ assertion -- it catches empty tables and failed loads -- and on Iceberg it's a metadata-cheap count.
for each dataset T:
  datasets:
    - include %
    - exclude %__dbt_tmp       # transient dbt build artifacts
    - exclude %__dbt_backup    # never cataloged as datasets
    - exclude %__dbt_backup%
  checks:
    - row_count > 0
```

`include %` matches every table in `iceberg.dbt`; the `exclude` lines drop dbt's transient build
artifacts so a scan mid-build doesn't false-fail. On Iceberg, `row_count` reads table metadata — it is
**not** a heavy `GROUP BY`, so there is no Trino OOM risk fanning it across the whole schema.


### A richer contract, for contrast — `music_silver.yml`

`baseline.yml` guarantees a *floor* of coverage everywhere. The specific mart/silver/gold files layer
**semantic bounds** on top. This is an excerpt of `soda/checks/music_silver.yml`, to show the range of
the check language — value ranges, key non-null, and `missing_percent(col) < 100` “not 100% NULL”
tripwires (a `min`/`max` check passes *vacuously* on an all-null column; the tripwire does not):

```yaml
checks for spotify_tracks:
  - row_count > 0
  - missing_count(track_id) = 0
  # audio-feature model scores are bounded [0, 1]
  - min(danceability) >= 0
  - max(danceability) <= 1
  - min(energy) >= 0
  - max(energy) <= 1
  # physical / catalog measures
  - min(tempo) >= 0
  - min(duration_ms) > 0
  - max(popularity) <= 100

checks for lastfm:
  - row_count > 0
  - missing_count(user_id) = 0
  - min(age) >= 0
  - max(age) <= 120
  - missing_percent(play_count) < 100   # tripwire: not 100% NULL
```

We do **not** run this file below — it targets the `weyland_music` silver source, and its findings are
**advisory** in the lab (dirty ages and 0-duration tracks live in datasets we downloaded and do not
control). The scan we run is the **baseline contract over the marts**, where every check is expected to
pass because the marts already survived dbt's build-time tests.


## Run the scan

We drive Soda through its **Python `Scan` API** (the same engine the CLI wraps). The four-line contract
is: name the data source, hand it the configuration YAML, hand it the checks (SodaCL) YAML, then
`execute()`. `execute()` returns an **exit code** — `0` all pass, `1` warnings, `2` check failures,
`3` an execution error — it does **not** raise when a check fails, so a failing contract comes back as
*data* we render, exactly as it should.

We embed the baseline checks as a string here so the cell is self-contained (it is the verbatim
`baseline.yml` shown above); a scheduled run points Soda at the file on disk instead.


In [3]:
from soda.scan import Scan

# the verbatim baseline.yml contract (embedded so this cell is self-contained; the Dagster op reads the file)
CHECKS_YAML = """
for each dataset T:
  datasets:
    - include %
    - exclude %__dbt_tmp
    - exclude %__dbt_backup
    - exclude %__dbt_backup%
  checks:
    - row_count > 0
"""

scan = Scan()
scan.set_data_source_name(DATA_SOURCE)
scan.add_configuration_yaml_str(CONFIG_YAML)
scan.add_sodacl_yaml_str(CHECKS_YAML)

exit_code = scan.execute()   # returns an exit code; does NOT raise on a check failure

# Soda's own exit-code legend -- print it so the number below is self-explanatory
_legend = {0: "all checks passed", 1: "warnings only", 2: "one or more checks FAILED", 3: "execution error"}
print(f"scan exit code : {exit_code}  ({_legend.get(exit_code, 'unknown')})")


scan exit code : 0  (all checks passed)


## Results — per-check outcomes as a frame

`scan.get_scan_results()` hands back the same JSON the Dagster op writes with `-srf` and emits to
DataHub: a `checks[]` array (each with a `name`, `table`, `outcome`, and the metric it measured) plus a
`metrics[]` array carrying the measured **value** for each. We join the two so every row shows the
**measured value next to its check** — then summarize pass / warn / fail, and, if anything failed,
surface it explicitly as a **finding** (a contract scan exists to make that visible).


In [4]:
import polars as pl

results = scan.get_scan_results()

# Fail CLOSED: exit 3 (execution error) or an empty checks[] means the scan never ran the contract --
# a connection/config failure, NOT a passing contract. Never let an absent result read as success.
_checks = results.get("checks", [])
if exit_code == 3 or not _checks:
    raise RuntimeError(
        f"Soda evaluated 0 checks (exit {exit_code}) -- a connection or config failure, not a held "
        f"contract. Refusing to render a false pass; check the data source / trino-noauth reachability."
    )

# measured value lives in the metrics[] array; each check references it by identity
_metric_value = {m.get("identity"): m.get("value") for m in results.get("metrics", [])}

def _measured(check):
    # value is either cross-referenced via the check's metric identities, or inline in diagnostics
    for mid in (check.get("metrics") or []):
        if mid in _metric_value:
            return _metric_value[mid]
    diag = check.get("diagnostics") or {}
    return diag.get("value")

rows = []
for c in results.get("checks", []):
    loc = c.get("location") or {}
    rows.append((
        c.get("table") or loc.get("table"),
        c.get("name") or c.get("definition"),
        (c.get("outcome") or "").upper(),
        _measured(c),
    ))

frame = pl.DataFrame(rows, schema=["table", "check", "outcome", "measured"], orient="row").sort(["outcome", "table"])
print(f"checks evaluated : {frame.height}")
frame


checks evaluated : 8


table,check,outcome,measured
str,str,str,i64
"""mart_artist_popularity""","""row_count > 0""","""PASS""",173993
"""mart_country_health""","""row_count > 0""","""PASS""",12732
"""mart_fma_genre_tree""","""row_count > 0""","""PASS""",164
"""mart_genre_audio_profile""","""row_count > 0""","""PASS""",113
"""mart_personality_by_country""","""row_count > 0""","""PASS""",52
"""mart_spotify_audio""","""row_count > 0""","""PASS""",89741
"""mart_state_health_trends""","""row_count > 0""","""PASS""",770
"""metricflow_time_spine""","""row_count > 0""","""PASS""",24472


In [5]:
from collections import Counter

# pass / warn / fail summary -- the headline the scan exists to produce
tally = Counter(c.get("outcome") for c in results.get("checks", []))
summary = pl.DataFrame(
    [("pass", tally.get("pass", 0)), ("warn", tally.get("warn", 0)), ("fail", tally.get("fail", 0))],
    schema=["outcome", "checks"], orient="row",
)
print(summary)

# if the contract held, say so; if anything failed, surface each finding explicitly (this is the point)
failures = [c for c in results.get("checks", []) if (c.get("outcome") or "").lower() == "fail"]
if not failures:
    print(f"\nCONTRACT HELD -- all {tally.get('pass', 0)} baseline checks passed over the marts.")
else:
    print(f"\nFINDINGS -- {len(failures)} check(s) failed (rendered as data, not an error):")
    for c in failures:
        loc = c.get("location") or {}
        print(f"  - {c.get('table') or loc.get('table')}: {c.get('name') or c.get('definition')} "
              f"(measured {_measured(c)})")


shape: (3, 2)
┌─────────┬────────┐
│ outcome ┆ checks │
│ ---     ┆ ---    │
│ str     ┆ i64    │
╞═════════╪════════╡
│ pass    ┆ 8      │
│ warn    ┆ 0      │
│ fail    ┆ 0      │
└─────────┴────────┘

CONTRACT HELD -- all 8 baseline checks passed over the marts.


### Reading the outcome

Every baseline check passed: each `iceberg.dbt.mart_*` table has rows, so the *published* contract
holds — independently of the dbt tests that gated the build. That is the reassurance an outside
contract buys: even a mart that shipped green from dbt is re-verified against the live table on every
scan, so a *post-publication* regression (a bad reload, a truncated table, an upstream source that went
empty) surfaces here rather than in a consumer's dashboard.

Had a check failed — say a mart came back empty — the row above would read `FAIL` with `measured 0`,
the summary would count it, and it would be listed as a finding. **No cell would crash**: a failing
contract is the scan doing its job, and the whole pipeline treats it as a red row to act on, not an
exception to swallow. In the lab, that same red outcome lands on the mart's **DataHub Assertions tab**
(the Dagster op emits it), which is the closest thing to a Soda UI without paying for Soda Cloud.


## When to reach for what

Data quality in this stack is layered, and the right tool depends on **when** you need the check to
fire and **how independent** it must be.

| reach for | when… | why |
|-----------|-------|-----|
| **dbt tests** (nb `40`) | the check should **block the build** — a bad mart must never ship | runs inside `dbt build`; a failing uniqueness/not-null/range test fails the build and nothing is published |
| **Soda** (this notebook) | you need an **independent contract** re-checked *after* publication, on a schedule, with per-check pass/fail emitted to the catalog | a separate engine over the *live* marts + gold sources; catches post-publication drift dbt's build-time tests can't see; freshness and anomaly checks; one-line SodaCL |
| **Great Expectations** (GE tier) | you need **rich per-column expectation suites** and a browsable data-docs site, and can afford heavier authoring | expectation-level granularity and generated documentation, at more setup cost than SodaCL's one-liners |

> **Gate the build with dbt tests. Re-assert the shipped result with a Soda contract scan. Reach for
> Great Expectations when a dataset needs column-level expectation suites and their data-docs.** All
> three read the *same* published marts — the tested `iceberg.dbt.mart_*` tables — and this notebook
> ran the middle one end-to-end, read-only, from inside the mesh via the `trino-noauth` proxy.
